# <span style="color:#F57C00">Risk Score and Priority: Which Districts Should Come First?</span>

**What this notebook does.** The what-if notebook told, for every district, how much green it needs to get cooler. This notebook asks *where it matters most*: it combines **how hot** a district is, **how many vulnerable people** live there and **how little developed** it is into one **risk score**, shows how much the ranking depends on our choices, and adds a **priority** that says where one hectare of green helps the most vulnerable people.

**Read this first.** A risk score is a *ranking tool*, not a measurement. The weights of the three parts are a choice; the notebook shows how much the ranking changes when the weights change, so that nobody reads a rank of 12 against 15 as a fact.

**The parts**

| Part | What happens |
| :--- | :--- |
| **1. The three components** | Hazard (heat), exposure (vulnerable people) and vulnerability (development), each as a percentile rank from 0 to 100 |
| **2. The risk score and its stability** | The equal-weight mean, and 500 random weight sets to see how stable the ranks are |
| **3. Priority** | Vulnerable people per hectare of green needed for -1 °C, for the districts that can reach it |
| **4. Maps, the top 20 and saving** | Maps, tables, and the complete `district_scorecard.csv` |

**Input** is the first part of `data/results/district_scorecard.csv` (from `whatif.ipynb`) and the processed tables. **Output** is the complete `district_scorecard.csv`. It covers the **829 districts** that have a reliable surface temperature; the other 144 districts (142 with a built-up area under 0.5 km² and the 2 reliable districts of Bayburt, see Part 1 of the what-if notebook) have no risk score.

**Run time** a few seconds.

# <span style="color:#F57C00">Part 1: The Three Components</span>

## Step 1: Libraries, settings and the tables

In [ ]:
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROCESSED = Path("../data/processed")
MODELS = Path("../data/models")
RESULTS = Path("../data/results")
SEED = 2025  # the random weight sets of Part 2
N_WEIGHTS = 500

scorecard = pd.read_csv(RESULTS / "district_scorecard.csv")
population = pd.read_csv(PROCESSED / "tuik_districts_population_clean.csv")
sege = pd.read_csv(PROCESSED / "sege_districts_2022.csv")
districts = gpd.read_file(PROCESSED / "turkey_districts_master.geojson")

assert (
    len(scorecard) == scorecard["district_code"].nunique() == 829
), "the scorecard of whatif.ipynb is needed"
print(
    f"scorecard {scorecard.shape} | population {population.shape} | sege {sege.shape} | districts {districts.shape}"
)

## Step 2: Hazard, exposure and vulnerability
- **Hazard** = how hot the district is: its mean surface temperature of the built-up area in 2023-2025 (`lst_recent`).
- **Exposure** = how many people are vulnerable to heat: the **number** of children aged 0-4 and of people aged 65 and over, in 2025 (urban population of TÜİK). It is a count, so big districts rank high; a share would favour tiny districts.
- **Vulnerability** = how little developed the district is: **minus** the SEGE-2022 development score (higher score = more developed, so a low score means more vulnerable).

Every component becomes a **percentile rank** from 0 to 100 over the 829 districts (100 = the worst: hottest, most vulnerable people, least developed). Ranks make the three components comparable without any assumption about their scale, and they only use the *order*: a logarithm of the count would give exactly the same ranks, so it is not needed.

In [ ]:
age_groups = [
    "total_0_4",
    "total_65_69",
    "total_70_74",
    "total_75_79",
    "total_80_84",
    "total_85_89",
    "total_90plus",
]
people = population[population["year"] == 2025].set_index("district_code")
sege_score = sege.set_index("district_code")["sege_score"]

scorecard["vulnerable_people"] = scorecard["district_code"].map(
    people[age_groups].sum(axis=1)
)
scorecard["hazard_pct"] = scorecard["lst_recent"].rank(pct=True) * 100
scorecard["exposure_pct"] = scorecard["vulnerable_people"].rank(pct=True) * 100
scorecard["vulnerability_pct"] = (-scorecard["district_code"].map(sege_score)).rank(
    pct=True
) * 100

COMPONENTS = ["hazard_pct", "exposure_pct", "vulnerability_pct"]
print(
    scorecard[["lst_recent", "vulnerable_people"] + COMPONENTS]
    .describe()
    .round(1)
    .loc[["min", "50%", "max"]]
    .to_string()
)
print("correlation between the components:")
print(scorecard[COMPONENTS].corr().round(2).to_string())

## Step 3: Tests
1. The count of vulnerable people is checked for 3 districts with an independent way of selecting the age columns (all columns of 0-4 and of 65 and over, found by name).
2. All components are between 0 and 100, there are no missing values, and the hottest district has a hazard of 100.
3. The vulnerability has the right direction: the less developed (lower SEGE score), the higher the vulnerability.

In [ ]:
def is_vulnerable_age(column):
    # total_0_4, and total_65_69 ... total_90plus
    if not column.startswith("total_") or column == "total_pop":
        return False
    first_age = int(column.split("_")[1][:2])  # 0, 65, 70, ...
    return first_age == 0 or first_age >= 65


by_name = [c for c in population.columns if is_vulnerable_age(c)]
assert sorted(by_name) == sorted(age_groups), "the age columns differ"
checked = scorecard.sample(3, random_state=0)
for _, district in checked.iterrows():
    row = population[
        (population["district_code"] == district["district_code"])
        & (population["year"] == 2025)
    ].iloc[0]
    assert district["vulnerable_people"] == row[by_name].sum()

assert (
    scorecard[COMPONENTS].notna().all().all()
    and scorecard[COMPONENTS].min().min() > 0
    and scorecard[COMPONENTS].max().max() == 100
)
assert scorecard.loc[scorecard["lst_recent"].idxmax(), "hazard_pct"] == 100
joined = scorecard["district_code"].map(sege_score)
assert (
    joined.notna().all()
    and scorecard["vulnerability_pct"].corr(joined, method="spearman") < -0.999
), "the less developed, the more vulnerable"
print(
    f"tests passed | {len(by_name)} age columns | the 3 districts checked have {checked['vulnerable_people'].astype(int).tolist()} vulnerable people"
)

**What we learned (Part 1)**
- **The three components point at different places** (maps in Part 4): the hazard is highest in the Southeast and in the interior of the Aegean, the vulnerability (low development) in the east and the Southeast, and the exposure (number of vulnerable people) in the big districts everywhere.
- **Hazard hardly depends on the other two** (correlation +0.19 and +0.05), but **exposure and vulnerability pull against each other** (-0.73): the districts with many vulnerable people are mostly the big, more developed ones. Because the components partly cancel, one high component is not enough: a risk score of 80 or more needs every component to be at least 40.

# <span style="color:#F57C00">Part 2: The Risk Score and How Stable It Is</span>

## Step 1: The risk score
The **risk score** is the mean of the three percentile ranks (0 to 100, equal weights: the simplest choice, and the one we can explain in one sentence). `risk_rank` 1 is the district with the highest risk.

## Step 2: What if the weights were different?
Equal weights are a choice. To see how much it matters we draw **500 random weight sets** (every set has three positive weights that sum to 1, drawn with a Dirichlet distribution, so every mix of the three components is equally likely) and calculate the ranks again with every set. For each district:
- `risk_rank_q05` and `risk_rank_q95`: the **raw** 5th and 95th percentile of its rank over the 500 sets;
- `risk_rank_low` and `risk_rank_high`: the same range **widened** to contain the rank of the equal weights (these are the exported bounds);
- `top_share`: the share of the weight sets in which it is among the 10% of the districts with the highest risk (the top 83 of 829);
- `risk_rank_*_only`: the rank if all the weight sat on one component (a policy scenario, saved in the scorecard).

A district with a narrow rank range and a `top_share` near 1 is high risk **in at least 90% of the sampled weight sets** — a frequency over the draws, not a probability of harm and not a claim about every possible weighting; a district with a wide range depends on what we find most important.

In [ ]:
scorecard["risk_score"] = scorecard[COMPONENTS].mean(axis=1)
scorecard["risk_rank"] = (
    scorecard["risk_score"].rank(ascending=False, method="first").astype(int)
)

weights = np.random.default_rng(SEED).dirichlet(
    [1, 1, 1], size=N_WEIGHTS
)  # 500 x 3, every row sums to 1
ranks = pd.DataFrame(scorecard[COMPONENTS].to_numpy() @ weights.T).rank(
    ascending=False, method="first"
)  # district x weight set
n_top = int(np.ceil(0.10 * len(scorecard)))  # the top 10%: 83 districts

low, high = np.round(np.percentile(ranks, [5, 95], axis=1))
widened = ((low > scorecard["risk_rank"]) | (high < scorecard["risk_rank"])).mean()
scorecard["risk_rank_low"] = np.minimum(low, scorecard["risk_rank"]).astype(
    int
)  # the range always contains the rank of the equal weights
scorecard["risk_rank_high"] = np.maximum(high, scorecard["risk_rank"]).astype(int)
scorecard["top_share"] = (ranks <= n_top).mean(axis=1)

scorecard["risk_rank_q05"] = np.percentile(
    ranks, 5, axis=1
)  # the raw quantiles, before widening
scorecard["risk_rank_q95"] = np.percentile(ranks, 95, axis=1)
for component in COMPONENTS:
    scorecard[f"risk_rank_{component.removesuffix('_pct')}_only"] = scorecard[component].rank(
        ascending=False, method="first"
    ).astype(
        int
    )  # a policy scenario: all the weight on one component

print(
    f"risk score: from {scorecard['risk_score'].min():.1f} to {scorecard['risk_score'].max():.1f} | rank ranges widened to contain the equal-weight rank: {widened:.1%} of the districts"
)
width = scorecard["risk_rank_high"] - scorecard["risk_rank_low"]
print(
    f"width of the rank range: median {width.median():.0f} places (of 829); in the top 10% in 90% or more of the weight sets: {(scorecard['top_share'] >= 0.9).sum()} districts, in some but not most: {((scorecard['top_share'] > 0) & (scorecard['top_share'] < 0.9)).sum()}, never: {(scorecard['top_share'] == 0).sum()}"
)

## Step 3: Tests
1. The risk score is between 0 and 100 and is the mean of the three components.
2. `risk_rank` uses every number from 1 to 829 exactly once.
3. `risk_rank_low` <= `risk_rank` <= `risk_rank_high`, and all ranks are between 1 and 829.
4. In every weight set exactly 83 districts are in the top 10%, so the mean of `top_share` over the districts is exactly 83 / 829.
5. The exported bounds stay at least as wide as the rounded raw quantiles (`risk_rank_low` ≤ round(`risk_rank_q05`), `risk_rank_high` ≥ round(`risk_rank_q95`)), and they contain the equal-weight rank.
6. A direct check of the sensitivity: with all the weight on one component, the ranks follow that component.

In [ ]:
assert scorecard["risk_score"].between(0, 100).all()
assert np.allclose(scorecard["risk_score"], scorecard[COMPONENTS].sum(axis=1) / 3)
assert sorted(scorecard["risk_rank"]) == list(range(1, 830))
assert (scorecard["risk_rank_low"] <= scorecard["risk_rank"]).all() and (
    scorecard["risk_rank"] <= scorecard["risk_rank_high"]
).all()
assert (
    scorecard["risk_rank_low"].min() >= 1 and scorecard["risk_rank_high"].max() <= 829
)
assert np.isclose(scorecard["top_share"].mean(), n_top / len(scorecard))
assert (scorecard["risk_rank_low"] <= scorecard["risk_rank_q05"].round()).all() and (
    scorecard["risk_rank_high"] >= scorecard["risk_rank_q95"].round()
).all(), "the widened bounds stay at least as wide as the rounded raw quantiles"
all_on_hazard = pd.DataFrame(
    scorecard[COMPONENTS].to_numpy() @ np.array([[1.0], [0.0], [0.0]])
).rank(ascending=False, method="first")[
    0
]  # the same calculation as above, with one weight set
assert (
    all_on_hazard.to_numpy()
    == scorecard["hazard_pct"].rank(ascending=False, method="first").to_numpy()
).all(), "all weight on the hazard must give the hazard ranking"
print("tests passed")

In [ ]:
top = scorecard.nsmallest(20, "risk_rank").iloc[
    ::-1
]  # the 20 highest risks, the first one on top of the figure
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
labels = top["district_name"] + " (" + top["city_name"] + ")"
axes[0].hlines(
    labels,
    top["risk_rank_low"],
    top["risk_rank_high"],
    color="#F57C00",
    lw=3,
    label="5-95% over the weight sets",
)
axes[0].plot(top["risk_rank"], labels, "o", color="black", label="equal weights")
axes[0].set_xlabel("risk rank (1 = highest risk)")
axes[0].set_title("The 20 highest risks and how stable their rank is")
axes[0].legend(loc="upper right", fontsize=8)

axes[1].hist(
    scorecard["top_share"], bins=20, color="#F57C00", edgecolor="black", linewidth=0.5
)
axes[1].set_xlabel("share of the weight sets in which the district is in the top 10%")
axes[1].set_ylabel("districts")
axes[1].set_title("How often is a district in the top 10%?")
plt.tight_layout()
plt.show()

**What we learned (Part 2)**
- **The highest risks are all in the Southeast.** All 10 districts with the highest risk are in Şanlıurfa (Siverek, Akçakale, Viranşehir, Eyyübiye, Harran, Ceylanpınar), Diyarbakır (Çınar, Silvan), Mardin (Derik) and Şırnak (Silopi): the hottest region, with large districts that are less developed.
- **The top is stable, the rest is not.** 12 districts are in the top 10% in at least 90% of the 500 weight sets, so they are high risk in at least 90% of the sampled weight sets; 272 districts are in it for some weights only, and 545 never. Outside the top the rank depends strongly on the weights (the rank range of a typical district is 427 places wide, of 829), so **do not read the exact rank**: use the group (top 10%, with `top_share`) and look at the range.
- Equal weights are a choice, not a finding. If a planner thinks that heat matters more than development, the weights change and so does the list; the ranges show how much.

# <span style="color:#F57C00">Part 3: Priority: Where Does One Hectare of Green Help Most?</span>

The risk score says where the risk is. It does not say where *action is cheap*. The **priority** combines the answer of the what-if notebook (how many hectares of green a district needs for -1 °C) with the number of vulnerable people:

**`people_per_hectare_1c` = vulnerable people / hectares of green needed for -1 °C**

A high value means that a few hectares of green would cool a place where many vulnerable people live. `priority_rank` 1 is the highest value. It is only given to districts that can reach -1 °C (`reachable_1c`); the others have no priority.

**Use it for ranking only, never as a number.** The numerator is an *urban* population (TÜİK) and the denominator is a built-up area that also includes villages and scattered buildings, so the ratio is not a real density of people per hectare; the order of the districts is what carries information.

**The target itself is uncertain, and the priority must say so.** The hectares above come from the **central** model. The what-if notebook trained 60 models on resampled cities and every draw has its own curves, so every draw can be solved for the same target. For each district this gives:
- `priority_reachability`: the share of the 60 draws in which it reaches -1 °C at all — draws that do not reach the target **stay in this frequency**; they are part of the answer, not a nuisance;
- `priority_rank_low` / `priority_rank_high`: the 5th and 95th percentile of its priority rank over the draws, **conditional on reaching the target** at least once (a district that never reaches it has none);
- `priority_top20_share`: the share of the draws in which it is among the 20 highest priorities.

These are frequencies over model draws — **model-sampling sensitivity**, not probabilities of harm and not a statement about any particular city. A district with a low top-20 share is one whose exact place in the list rests on which cities the model happened to see.

In [ ]:
scorecard["people_per_hectare_1c"] = (
    scorecard["vulnerable_people"] / scorecard["hectares_1c"]
)  # empty where -1 °C is not reachable
scorecard["priority_rank"] = (
    scorecard["people_per_hectare_1c"]
    .rank(ascending=False, method="first")
    .astype("Int64")
)

n_priority = scorecard["priority_rank"].notna().sum()
assert (
    n_priority == scorecard["reachable_1c"].sum()
), "a priority exactly for the reachable districts"
assert sorted(scorecard["priority_rank"].dropna()) == list(range(1, n_priority + 1))
assert (scorecard.loc[~scorecard["reachable_1c"], "people_per_hectare_1c"].isna()).all()

print(
    f"priority for {n_priority} districts; {len(scorecard) - n_priority} cannot reach -1 °C and have none"
)
density = (
    scorecard["district_code"].map(people["total_pop"]) / scorecard["urban_km2"]
)  # urban population per km² of measured built-up area (a check, not a column of the scorecard)
first = scorecard["priority_rank"] <= 20
print(
    f"the 20 highest priorities have a median of {density[first].median():,.0f} people per km² of built-up area, all districts {density.median():,.0f}; {(scorecard.loc[first, 'urban_km2'] < 1).sum()} of the 20 have less than 1 km² of built-up area"
)
print(
    f"the priority follows the density of people: Spearman correlation of the two ranks {scorecard['priority_rank'].astype(float).corr(density.rank(ascending=False), method='spearman'):+.2f}, and the points of green needed only {scorecard['priority_rank'].astype(float).corr(scorecard['green_needed_1c_pts'].rank(), method='spearman'):+.2f}"
)
both = scorecard[
    (scorecard["risk_rank"] <= n_top) & (scorecard["priority_rank"] <= n_top)
]
print(
    f"in the top {n_top} of the risk AND of the priority: {len(both)} districts | correlation of the two ranks: {scorecard['risk_rank'].corr(scorecard['priority_rank'].astype(float), method='spearman'):+.2f} (Spearman)"
)

### The uncertainty of the priority: every draw, not only the central model

The hectares of the table above come from the **central** model. The what-if notebook trained 60 models on resampled cities; every draw gives its own curve, so every draw can be solved for the same target. For each district this gives:

- `priority_reachability`: the share of the 60 draws in which it reaches -1 °C at all. Draws that do not reach the target **stay in this frequency** — they are part of the answer, not a nuisance;
- `priority_rank_low` / `priority_rank_high`: the 5th and 95th percentile of its priority rank over the draws, **conditional on reaching the target** at least once (a district that never reaches it has no bounds);
- `priority_top20_share`: the share of the draws in which it is among the 20 highest priorities.

These are frequencies over model draws — **model-sampling sensitivity**, not probabilities of harm, and not a statement about any particular city. A district with a low top-20 share is one whose exact place in the list rests on which cities the model happened to see.

In [ ]:
bank = np.load(MODELS / "bootstrap_effects.npz")
assert bank["bootstrap_unit"].item() == "city", "this needs the city bootstrap of whatif.ipynb"
assert "max_green" in bank, "the bank is old; run whatif.ipynb again"
effects_boot = bank["effects"].astype(float)  # draw x district x step
grid_bank = bank["grid"]
max_green = float(bank["max_green"])
zero_step = int(np.flatnonzero(grid_bank == 0)[0])
assert list(bank["district_code"]) == list(scorecard["district_code"]), "the bank and the scorecard must be in the same order"

steps = grid_bank[zero_step:] / 100  # the requested change of the steps 0 ... +30
share = scorecard["green_share"].to_numpy()
applied = np.clip(share[:, None] + steps, 0, max_green) - share[:, None]  # the change really applied
area = scorecard["urban_km2"].to_numpy()
people_all = scorecard["vulnerable_people"].to_numpy()

needs = np.full((len(effects_boot), len(scorecard)), np.nan)
for b, curve in enumerate(effects_boot[:, :, zero_step:]):
    reached = -curve >= 1.0  # the target: -1 °C
    first = reached.argmax(axis=1)
    rows = np.arange(len(scorecard))
    needs[b] = np.where(reached.any(axis=1), applied[rows, first], np.nan)

hectares_draw = needs * area[None, :]
priority_draw = np.where(hectares_draw > 0, people_all[None, :] / hectares_draw, np.nan)
ranks_draw = pd.DataFrame(priority_draw).rank(axis=1, ascending=False, method="first")

scorecard["priority_reachability"] = np.isfinite(needs).mean(axis=0)
scorecard["priority_rank_low"] = pd.array(np.round(ranks_draw.quantile(0.05)), dtype="Int64")
scorecard["priority_rank_high"] = pd.array(np.round(ranks_draw.quantile(0.95)), dtype="Int64")
scorecard["priority_top20_share"] = (ranks_draw <= 20).mean(axis=0).to_numpy()

top20 = scorecard.nsmallest(20, "priority_rank")
print(
    f"reaching -1 °C: {scorecard['priority_reachability'].mean():.1%} of the district-draw pairs on average; "
    f"{(scorecard['priority_reachability'] == 0).sum()} districts do not reach it in any of the 60 draws"
)
print(
    f"of the current top 20, {(top20['priority_top20_share'] <= 0.5).sum()} are in the top 20 in at most half of the draws: "
    + ", ".join(
        f"{row.district_name} ({row.priority_top20_share:.0%})"
        for row in top20.itertuples()
        if row.priority_top20_share <= 0.5
    )
)
assert scorecard[["priority_reachability", "priority_top20_share"]].min().min() >= 0
assert scorecard[["priority_reachability", "priority_top20_share"]].max().max() <= 1
print("rank bounds are conditional on reaching the target; unreachable draws stay in the reachability and top-20 frequencies")

**What we learned (Part 3)**
- **785 districts have a priority** (the ones that can reach -1 °C). The top of the list is a mix: small towns, mostly on the Black Sea (Kumru, Perşembe, Maçka, Vakfıkebir, Asarcık, Mesudiye; 18 of the top 40), where a few hectares would reach -1 °C, and Southeastern districts that are also high risk (Kulp, Kızıltepe, Bağlar; the lists of risk and priority share 16 of their top 83 districts).
- **Be careful with the priority: it is mostly a density.** `people_per_hectare_1c` is the number of vulnerable people divided by the hectares needed, and the hectares are the points of green times the *measured built-up area*. The priority follows the density of people in the built-up area (Spearman correlation +0.86) much more than the points of green that are needed (+0.38). The 20 highest priorities have a median of 26,100 people per km² of built-up area, against 8,600 for all districts, and 13 of the 20 have less than 1 km² of measured built-up area. Either these places are really very dense, or the built-up area of a small town is measured too small (30 m pixels and a small area: a handful of pixels decide). The notebook cannot tell which, so a high priority is a reason to **look at the place first**, not a result to act on.
- The priority is also silent on the risk: a cool, developed small town can have a high priority. It is meant to be read **together with the risk score**, for example "high risk and high priority": Cizre, Kulp, Kızıltepe and Harran.
- **The exact rank also depends on which cities the model has seen** (the stability lines printed above): read `priority_reachability` and `priority_top20_share` next to the rank, and remember that the rank bounds are conditional on reaching the target at all. A district that is in the top 20 in only some of the 60 draws is a candidate to look at, not a fixed place in the list.

# <span style="color:#F57C00">Part 4: Maps, the Top 20 and Saving</span>

## Step 1: Maps
The first three maps show the components, the fourth the risk score. Grey districts have no reliable surface temperature and therefore no score.

In [ ]:
shapes = districts[["district_code", "geometry"]].merge(
    scorecard[["district_code"] + COMPONENTS + ["risk_score"]],
    on="district_code",
    how="left",
    validate="one_to_one",
)
fig, axes = plt.subplots(2, 2, figsize=(13, 7.5))
panels = [
    ("hazard_pct", "Hazard: how hot"),
    ("exposure_pct", "Exposure: vulnerable people"),
    ("vulnerability_pct", "Vulnerability: low development"),
    ("risk_score", "Risk score"),
]
for ax, (column, title) in zip(axes.ravel(), panels):
    shapes.plot(
        column=column,
        cmap="YlOrRd",
        vmin=0,
        vmax=100,
        linewidth=0.1,
        edgecolor="white",
        ax=ax,
        missing_kwds={"color": "#dddddd"},
    )
    ax.set_title(title)
    ax.set_axis_off()
plt.tight_layout()
plt.show()

## Step 2: The top 20
The 20 districts with the highest risk, and the 20 with the highest priority (with their risk rank, to see how far the two lists overlap). `green_needed_1c_pts` is the points of green share for -1 °C from the what-if notebook.

In [ ]:
show = [
    "district_name",
    "city_name",
    "lst_recent",
    "vulnerable_people",
    "risk_score",
    "risk_rank_low",
    "risk_rank_high",
    "top_share",
]
print("The 20 highest risks")
print(
    scorecard.nsmallest(20, "risk_rank")
    .set_index("risk_rank")[show]
    .round(2)
    .to_string()
)
print()
show = [
    "district_name",
    "city_name",
    "vulnerable_people",
    "green_needed_1c_pts",
    "hectares_1c",
    "risk_rank",
    "priority_reachability",
    "priority_top20_share",
]
print("The 20 highest priorities (only for ranking)")
print(
    scorecard.nsmallest(20, "priority_rank")
    .set_index("priority_rank")[show]
    .round(1)
    .to_string()
)

## Step 3: Save
The complete scorecard goes back to `data/results/district_scorecard.csv`, with the columns in a fixed order. **What the columns mean:**

| Group | Columns | Meaning |
| :--- | :--- | :--- |
| keys | `district_code`, `city_code`, `district_name`, `city_name`, `region` | Join on the codes, never on the names (100 district names repeat) |
| state | `latest_year`, `urban_km2`, `green_share`, `lst_recent` | The latest reliable summer; built-up area (km²); green share of the built-up area (0-1); mean surface temperature 2023-2025 (°C) |
| what-if | `effect_10pts`, `_low`, `_high` | Change of the surface temperature for +10 points of green share (°C): central value and 5-95% range |
| scenario support | `support_10pts`, `support_30pts`, `small_area` | Flags from what-if Part 5: is the +10/+30 scenario close to comparable observations in the same region; `small_area` warns when the measured built-up area is under 1 km² |
| heat target -1 °C | `green_needed_1c_pts`, `hectares_1c`, each with `_low` (optimistic end) and `_high`; `reachable_1c` | Points of green share and hectares for -1 °C; empty where the target is not reached within +30 points |
| to the city median | `green_needed_median_pts`, `hectares_median`, `reachable_median` | The same for closing the gap to the median district of the city; empty for districts that are not hotter than the median |
| risk | `hazard_pct`, `exposure_pct`, `vulnerability_pct`, `risk_score`, `risk_rank`, `risk_rank_q05`, `risk_rank_q95`, `risk_rank_low`, `risk_rank_high`, `top_share`, `risk_rank_*_only` | The three components (percentile ranks 0-100, higher = worse), their mean and its rank; the raw 5-95% quantiles over 500 random weight sets and the same bounds **widened** to contain the equal-weight rank (the exports use the widened ones); the share of the sets in the top 10%; the rank under a single-component policy weight |
| priority | `vulnerable_people`, `people_per_hectare_1c`, `priority_rank`, `priority_reachability`, `priority_rank_low`, `priority_rank_high`, `priority_top20_share` | Children aged 0-4 and people aged 65+ (2025); people per hectare needed for -1 °C (for ranking only); rank among the districts that can reach -1 °C; over the 60 bootstrap draws: the share that reaches the target, the top-20 frequency, and rank bounds **conditional on reaching** the target |

In [ ]:
ORDER = [
    "district_code",
    "city_code",
    "district_name",
    "city_name",
    "region",
    "latest_year",
    "urban_km2",
    "green_share",
    "lst_recent",
    "effect_10pts",
    "green_needed_1c_pts",
    "hectares_1c",
    "effect_10pts_low",
    "green_needed_1c_pts_low",
    "hectares_1c_low",
    "effect_10pts_high",
    "green_needed_1c_pts_high",
    "hectares_1c_high",
    "reachable_1c",
    "green_needed_median_pts",
    "hectares_median",
    "reachable_median",
    "small_area",
    "support_10pts",
    "support_30pts",
    "vulnerable_people",
    "hazard_pct",
    "exposure_pct",
    "vulnerability_pct",
    "risk_score",
    "risk_rank",
    "risk_rank_q05",
    "risk_rank_q95",
    "risk_rank_low",
    "risk_rank_high",
    "top_share",
    "risk_rank_hazard_only",
    "risk_rank_exposure_only",
    "risk_rank_vulnerability_only",
    "people_per_hectare_1c",
    "priority_rank",
    "priority_reachability",
    "priority_rank_low",
    "priority_rank_high",
    "priority_top20_share",
]
assert set(ORDER) == set(scorecard.columns), "a column is missing or new"
assert (
    scorecard["district_code"].is_unique
    and len(scorecard) == 829
    and scorecard["city_code"].nunique() == 80
)

scorecard[ORDER].to_csv(RESULTS / "district_scorecard.csv", index=False)  # full precision: rounding can flip near-tied ranks
print(
    f"district_scorecard.csv: {len(scorecard)} rows, {len(ORDER)} columns, {(RESULTS / 'district_scorecard.csv').stat().st_size / 1e3:.0f} KB"
)

**What we learned (Part 4)**
- **The complete scorecard** has 829 rows and 45 columns: the state of every district, its what-if with a range, the two heat targets, the scenario-support flags, the risk score with its rank range and the priority with its reachability and top-20 frequencies. It is the file for a map or a table in an application; `what_if_curves.csv` holds the curves for a slider.
- **The map of the risk score** is dominated by the Southeast: 60 of the 83 districts of the top 10% are there (mean risk score 76, against 37 to 54 in the other regions). The Mediterranean (10 districts) and the Aegean (8) follow; the Black Sea has the lowest risk. The grey districts have no reliable surface temperature and no score.
- **What the score is not:** it does not know about shade, water, cooling centers, housing quality or the air temperature, it uses the *urban* population of TÜİK, and it is calculated with the surface temperature of 2023-2025. It is a first filter for where to look, not a decision.